# 05d: out-of-fold check of the B0 + B2 average

Decides whether averaging b0-v1-mm-flip and b2-v1-mm-flip is worth an inference run.
No training, no GPU: it only reads the out-of-fold predictions both runs saved.

Rule, fixed before running: go if the 50/50 average beats B0 alone in at least 4 of 5 folds
on silver macro AUC and the mean fold difference is positive. Gold is reported as a second
check, with a bootstrap interval, but it does not decide.

Inputs: competition data, output of 04 (silver_scores.csv), output of 05a-train-mm-flip,
output of 05a-train-mm-flip-b2.
Settings: CPU is enough, Internet off.

# Cell 1: find both runs and load their out-of-fold predictions

In [ ]:
import os, glob, json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

INPUT = "/kaggle/input"
ROOT = f"{INPUT}/competitions/rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
SILVER_POS = 3.0
BASE, OTHER = "b0-v1-mm-flip", "b2-v1-mm-flip"
FOLD_COLS = [f"f{f}" for f in range(5)]

def find_all(pattern):
    hits = []
    for top in os.listdir(INPUT):
        if top != "competitions":
            hits += glob.glob(f"{INPUT}/{top}/**/{pattern}", recursive=True)
    return sorted(hits)

runs = {}
for p in find_all("oof_summary.json"):
    exp = json.load(open(p))["exp"]
    runs[exp] = os.path.dirname(p)
print("runs found:", runs)
for exp in [BASE, OTHER]:
    if exp not in runs:
        raise RuntimeError(f"{exp} not found: attach its 05a output")
    n = len(glob.glob(f"{runs[exp]}/oof_f*.csv"))
    if n != 5:
        raise RuntimeError(f"{exp}: {n} oof files instead of 5")

def load_oof(exp):
    df = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f"{runs[exp]}/oof_f*.csv"))], ignore_index=True)
    return df.set_index("StudyInstanceUID")[LABELS].sort_index()

P = {exp: load_oof(exp) for exp in [BASE, OTHER]}
ids = P[BASE].index
if not ids.equals(P[OTHER].index) or ids.duplicated().any():
    raise RuntimeError("the two runs do not cover the same studies")

folds = {exp: pd.read_csv(f"{runs[exp]}/folds.csv").set_index("StudyInstanceUID")["fold"].loc[ids]
         for exp in [BASE, OTHER]}
if not folds[BASE].equals(folds[OTHER]):
    raise RuntimeError("the two runs used different folds")
fold = folds[BASE].values

silver = pd.read_csv(find_all("silver_scores.csv")[0]).set_index("StudyInstanceUID")[LABELS].loc[ids]
ysil = (silver >= SILVER_POS).astype(int)
print("studies:", len(ids), "| per fold:", np.bincount(fold).tolist())

def blend(w):
    return (1 - w) * P[BASE] + w * P[OTHER]

def macro_auc(y, p):
    aucs = {}
    for k in LABELS:
        t = y[k].values
        if 0 < t.sum() < len(t):
            aucs[k] = roc_auc_score(t, p[k].values)
    return float(np.mean(list(aucs.values()))), aucs

# Cell 2: silver macro AUC per fold

w_b2 is the weight on B2: 0 is B0 alone, 1 is B2 alone, 0.5 the plain average.
Row 0 should match the B0 training log (0.838, 0.858, 0.850, 0.828, 0.823) and row 1 the
B2 log (0.845, 0.860, 0.848, 0.825, 0.818). The 0.3 and 0.7 rows are for information only.

In [ ]:
rows = []
for w in [0.0, 0.3, 0.5, 0.7, 1.0]:
    pb = blend(w)
    r = {"w_b2": w}
    for f in range(5):
        m = fold == f
        r[f"f{f}"] = macro_auc(ysil[m], pb[m])[0]
    r["fold_mean"] = float(np.mean([r[c] for c in FOLD_COLS]))
    r["pooled"] = macro_auc(ysil, pb)[0]
    rows.append(r)
tab = pd.DataFrame(rows).set_index("w_b2")
print(tab.round(4).to_string())

diff = tab.loc[0.5, FOLD_COLS] - tab.loc[0.0, FOLD_COLS]
wins = int((diff > 0).sum())
print("\naverage minus B0 per fold:", diff.round(4).to_dict())
print(f"average wins {wins} of 5 folds, mean difference {diff.mean():+.4f}")

per = pd.DataFrame({"b0": macro_auc(ysil, blend(0.0))[1], "b2": macro_auc(ysil, blend(1.0))[1],
                    "avg": macro_auc(ysil, blend(0.5))[1]})
print("\nsilver out-of-fold per finding:")
print(per.round(3).to_string())

# Cell 3: gold check (58 studies)

Per-finding gold AUC for B0, B2 and the average, then a paired bootstrap of the macro
difference (average minus B0). The interval shows how much of the gold difference is noise.

In [ ]:
train = pd.read_csv(f"{ROOT}/train.csv")
gold = train[train[LABELS].notna().all(axis=1)].set_index("StudyInstanceUID")[LABELS].astype(int)
gold = gold.loc[gold.index.intersection(ids)]
pg = {name: blend(w).loc[gold.index] for name, w in [("b0", 0.0), ("b2", 1.0), ("avg", 0.5)]}
gt = pd.DataFrame({name: macro_auc(gold, p)[1] for name, p in pg.items()})
print("gold studies:", len(gold))
print(gt.round(3).to_string())
print("gold macro:", gt.mean().round(4).to_dict())

rng = np.random.default_rng(0)
boot = []
for _ in range(1000):
    i = rng.integers(0, len(gold), len(gold))
    g = gold.iloc[i]
    boot.append(macro_auc(g, pg["avg"].iloc[i])[0] - macro_auc(g, pg["b0"].iloc[i])[0])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"gold macro, average minus B0: {gt['avg'].mean() - gt['b0'].mean():+.4f} "
      f"(bootstrap 95% interval {lo:+.4f} to {hi:+.4f})")

# Cell 4: decision

In [ ]:
go = wins >= 4 and diff.mean() > 0
print("decision:", "GO, build the ensemble 05b" if go else "SKIP, keep B0 alone")
json.dump({"base": BASE, "other": OTHER, "silver_fold": tab.round(5).reset_index().to_dict("records"),
           "wins": wins, "mean_fold_diff": float(diff.mean()),
           "gold_macro": gt.mean().round(5).to_dict(), "gold_diff_ci": [float(lo), float(hi)], "go": bool(go)},
          open("/kaggle/working/ensemble_oof.json", "w"), indent=1)
print("saved ensemble_oof.json")